## Install Library

In [ ]:
# Install the YOLOv8 package
!pip install ultralytics

# Clear the output and check the GPU status
import ultralytics
ultralytics.checks()

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.0/112.6 GB disk)


## Mount with Google Drive




In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Copy data.yaml to each algorithm's dataset

In [ ]:
import yaml
import os
import shutil

base_path = '/content/drive/MyDrive/Thesis_Pothole_Project'
baseline_yaml = f'{base_path}/Dataset_1_Baseline/data.yaml'
datasets = ['Dataset_1_Baseline', 'Dataset_2_CLAHE', 'Dataset_3_AGCWD', 'Dataset_4_Retinex']

for dataset in datasets:
    yaml_path = f'{base_path}/{dataset}/data.yaml'

    # If the yaml file doesn't exist in the enhanced folders, copy it from Baseline first!
    if not os.path.exists(yaml_path):
        shutil.copy(baseline_yaml, yaml_path)
        print(f"Copied data.yaml to {dataset}")

    # Read the yaml file
    with open(yaml_path, 'r') as file:
        data = yaml.safe_load(file)

    # Overwrite with the absolute Google Drive paths
    data['train'] = f'{base_path}/{dataset}/train/images'
    data['val'] = f'{base_path}/{dataset}/valid/images'
    data['test'] = f'{base_path}/{dataset}/test/images'

    # Save the fixed yaml file
    with open(yaml_path, 'w') as file:
        yaml.dump(data, file, sort_keys=False)

print("\n✅ All data.yaml files successfully copied and updated for Google Colab!")

Copied data.yaml to Dataset_2_CLAHE
Copied data.yaml to Dataset_3_AGCWD
Copied data.yaml to Dataset_4_Retinex

✅ All data.yaml files successfully copied and updated for Google Colab!


#Combine class to single "Pothole" Class

In [ ]:
import os
import glob
import yaml
from tqdm import tqdm

base_path = '/content/drive/MyDrive/Thesis_Pothole_Project'
datasets = ['Dataset_1_Baseline', 'Dataset_2_CLAHE', 'Dataset_3_AGCWD', 'Dataset_4_Retinex']

print("Starting Universal Class Merger...")

for dataset in datasets:
    print(f"\nProcessing {dataset}...")

    # 1. Sweep and fix every single .txt label file
    label_search_path = os.path.join(base_path, dataset, '**', 'labels', '*.txt')
    label_files = glob.glob(label_search_path, recursive=True)

    for file_path in tqdm(label_files, desc="Updating labels to class 0"):
        with open(file_path, 'r') as file:
            lines = file.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            # YOLO format is: class_id x_center y_center width height
            if len(parts) >= 5:
                parts[0] = '0' # Force the class ID to be 0
                new_line = ' '.join(parts) + '\n'
                new_lines.append(new_line)

        # Overwrite the old file with the corrected 0 labels
        with open(file_path, 'w') as file:
            file.writelines(new_lines)

    # 2. Rewrite the data.yaml file to erase the old classes and set 'pothole' as the single class
    yaml_path = os.path.join(base_path, dataset, 'data.yaml')
    data = {} # Initialize with an empty dictionary

    # Read existing data if the file exists, otherwise use the empty dictionary
    if os.path.exists(yaml_path):
        try:
            with open(yaml_path, 'r') as file:
                data = yaml.safe_load(file)
            if data is None: # Handle empty YAML file if it exists
                data = {}
        except Exception as e:
            print(f"Warning: Could not load data.yaml at {yaml_path}: {e}. Reinitializing data.")
            data = {} # Reset to empty if loading fails
    else:
        print(f"Info: data.yaml not found at {yaml_path}. Creating a new one.")

    # Hardcode the configuration to be 1 class named 'pothole'
    data['nc'] = 1
    data['names'] = ['pothole']

    # Ensure the parent directory exists before writing
    os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

    with open(yaml_path, 'w') as file:
        yaml.dump(data, file, sort_keys=False)

print("\n✅ Success! All datasets have been successfully merged into a single 'pothole' class.")

Starting Universal Class Merger...

Processing Dataset_1_Baseline...


Updating labels to class 0: 100%|██████████| 6108/6108 [21:34<00:00,  4.72it/s]



Processing Dataset_2_CLAHE...


Updating labels to class 0: 100%|██████████| 6101/6101 [20:48<00:00,  4.89it/s]



Processing Dataset_3_AGCWD...


Updating labels to class 0: 100%|██████████| 6101/6101 [21:08<00:00,  4.81it/s]



Processing Dataset_4_Retinex...


Updating labels to class 0: 100%|██████████| 6101/6101 [25:42<00:00,  3.96it/s]



✅ Success! All datasets have been successfully merged into a single 'pothole' class.


# Check Training Dataset exist

In [ ]:
import os

# Path to the data.yaml file that YOLO is looking for
yaml_file_path = '/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml'

# Check if the file exists
if os.path.exists(yaml_file_path):
    print(f"✅ The file '{yaml_file_path}' exists.")
    # Try to read and print its content to check for corruption
    try:
        with open(yaml_file_path, 'r') as f:
            content = f.read()
            print("File content:\n" + content)
    except Exception as e:
        print(f"❌ Error reading file content: {e}")
else:
    print(f"❌ The file '{yaml_file_path}' does NOT exist after processing in the previous cell.")



✅ The file '/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml' exists.
File content:
train: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/train/images
val: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/valid/images
test: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/test/images
nc: 1
names:
- pothole
roboflow:
  workspace: yi-pin-wong
  project: pothole-detection-lowlight
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/yi-pin-wong/pothole-detection-lowlight/dataset/1



# Uninstall albumentations to ensure no lightning adjustment in training

In [ ]:
!pip uninstall albumentations -y

Found existing installation: albumentations 2.0.8
Uninstalling albumentations-2.0.8:
  Successfully uninstalled albumentations-2.0.8


# Baseline Training with nano model

In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data=/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml epochs=50 imgsz=640 batch=16 project=/content/drive/MyDrive/Thesis_Pothole_Project/Training_Results name=Baseline_Model_YOLOv8n hsv_h=0.0 hsv_s=0.0 hsv_v=0.0 bgr=0.0 mixup=0.0 copy_paste=0.0 flipud=0.0 degrees=0.0 perspective=0.0 shear=0.0 scale=0.3 translate=0.2 fliplr=0.5 erasing=0.4

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Baseline_Model_YOLOv8n, nbs=64, nms=False, opset=None, opti

## Final Performance Evaluation (Test Set)


In [ ]:
!yolo task=detect mode=val \
model=/content/drive/MyDrive/Thesis_Pothole_Project/Training_Results/Baseline_Model_YOLOv8n/weights/best.pt \
data=/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml \
split=test \
project=/content/drive/MyDrive/Thesis_Pothole_Project/Test_Results \
name=Baseline_Model_YOLOv8n_Test_Results

Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 0.1±0.0 MB/s, size: 39.7 KB)
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/test/labels.cache... 607 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 607/607 115.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 1.9it/s 19.6s
                   all        607       1356      0.826      0.768      0.843      0.524
Speed: 1.4ms preprocess, 3.9ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/drive/MyDrive/Thesis_Pothole_Project/Test_Results/Baseline_Model_YOLOv8n_Test_Results
💡 Learn more at https://docs.ultralytics.com/modes/val


## Evaluation for Baseline 1: Low-Light vs. Normal-Light Test Sets
To confirm that the model's detection performance is better in daytime conditions compared to nighttime conditions.

In [ ]:
import yaml
from ultralytics import YOLO

# 1. Load original data.yaml to keep class names and dataset structure intact
original_yaml_path = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml"
with open(original_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# 2. Override Only the test path to point to sorted normal Light images
data_config['test'] = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/images"

# 3. Save as a new configuration file specifically for normal light testing
day_yaml_path = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data_day.yaml"
with open(day_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"Created new normal light config at: {day_yaml_path}")

# 4. Load the trained baseline model weights
model = YOLO("/content/drive/MyDrive/Thesis_Pothole_Project/Training_Results/Baseline_Model_YOLOv8n/weights/best.pt")

# 5. Validate the model on the low-light test split using the new YAML
model.val(
    data=day_yaml_path,    # Use the newly created day configuration
    split="test",          # Tell YOLO to evaluate the 'test' split from that config

    # Standard save locations
    project="/content/drive/MyDrive/Thesis_Pothole_Project/Test_Results",
    name="Normal-Light_Baseline_Model_YOLOv8n_Test_Results",
    exist_ok=True
);

Created new normal light config at: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data_day.yaml
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.9±0.6 ms, read: 0.1±0.1 MB/s, size: 65.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/labels... 294 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 294/294 3.4it/s 1:27
val: New cache created: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_day/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.0it/s 6.3s
                   all        294        594 

In [ ]:
import yaml
from ultralytics import YOLO

# 1. Load original data.yaml to keep class names and dataset structure intact
original_yaml_path = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data.yaml"
with open(original_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# 2. Override Only the test path to point to sorted Low-Light images
data_config['test'] = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_night/images"

# 3. Save as a new configuration file specifically for low-light testing
night_yaml_path = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data_night.yaml"
with open(night_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"Created new low-light config at: {night_yaml_path}")

# 4. Load the trained baseline model weights
model = YOLO("/content/drive/MyDrive/Thesis_Pothole_Project/Training_Results/Baseline_Model_YOLOv8n/weights/best.pt")

# 5. Validate the model on the low-light test split using the new YAML
model.val(
    data=night_yaml_path,  # Use the newly created night configuration
    split="test",          # Tell YOLO to evaluate the 'test' split from that config

    # Standard save locations
    project="/content/drive/MyDrive/Thesis_Pothole_Project/Test_Results",
    name="Low-Light_Baseline_Model_YOLOv8n_Test_Results",
    exist_ok=True
);

Created new low-light config at: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/data_night.yaml
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.3 ms, read: 0.3±0.7 MB/s, size: 20.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline/dataset_split/test_night/labels.cache... 289 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 289/289 52.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.5it/s 7.7s
                   all        289        571      0.814      0.782      0.834      0.483
Speed: 2.1ms preprocess, 5.0ms inference, 0.0ms loss, 1.9ms postprocess 

## Baseline 2 Training (150 epochs)

In [ ]:
import yaml
import os
import shutil
from ultralytics import YOLO

# 1. Define base directory and paths
base_dir = "/content/drive/MyDrive/Thesis_Pothole_Project"
source_ds = f"{base_dir}/Dataset_1_Baseline"
target_ds = f"{base_dir}/Dataset_1_Baseline2"

print("Step 1: Setting up Dataset_1_Baseline2...")

# Duplicate Dataset_1_Baseline to Dataset_1_Baseline2 if it does not exist yet
if not os.path.exists(target_ds):
    print(" copying files from Dataset_1_Baseline to Dataset_1_Baseline2...")
    shutil.copytree(source_ds, target_ds)
    print("Dataset_1_Baseline2 created successfully!")
else:
    print("Dataset_1_Baseline2 already exists. Proceeding to training...")

# 2. Generate a dedicated YAML config file inside Dataset_1_Baseline2
original_yaml_path = os.path.join(target_ds, "data.yaml")
new_yaml_path = os.path.join(target_ds, "data_baseline2.yaml")

with open(original_yaml_path, 'r') as f:
    base_config = yaml.safe_load(f)

# Point train and val paths to Dataset_1_Baseline2
base_config['train'] = os.path.join(target_ds, 'train', 'images')
base_config['val'] = os.path.join(target_ds, 'valid', 'images')

# Remove test path during training to prevent data leakage
if 'test' in base_config:
    del base_config['test']

with open(new_yaml_path, 'w') as f:
    yaml.dump(base_config, f)

print(f"Generated new config file at: {new_yaml_path}")

# 3. Load Model and Start Training
print("\nStep 2: Launching 150-Epoch Baseline Training...")

model = YOLO("yolov8n.pt")

model.train(
    data=new_yaml_path,
    epochs=150,                  # 150 epochs for sufficient training
    patience=15,                 # Match early stopping threshold
    imgsz=640,
    batch=16,
    project=f"{base_dir}/Training_Results",
    name="Optimized_Model_Baseline2_YOLOv8n",
    exist_ok=True,

    hsv_h=0.0,                   # Disable Hue shifting
    hsv_s=0.0,                   # Disable Saturation shifting
    hsv_v=0.0,                   # Disable Brightness/Value shifting
    mosaic=0.0                   # Disable image scrambling
)

print("\nTraining finished for Dataset_1_Baseline2!")

Step 1: Setting up Dataset_1_Baseline2...
Dataset_1_Baseline2 already exists. Proceeding to training...
Generated new config file at: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/data_baseline2.yaml

Step 2: Launching 150-Epoch Baseline Training...
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline2/data_baseline2.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5